# Zomato Delhi — Data Cleaning

Làm sạch dữ liệu gốc Zomato (Ấn Độ). Chạy file này ĐẦU TIÊN, xuất ra data/clean/zomato_clean.csv cho 2 notebook sau dùng.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)

## ĐƯỜNG DẪN

In [2]:
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd().resolve().parent
RAW_PATH = BASE_DIR / 'data' / 'raw' / 'order_history_kaggle_data.csv'
CLEAN_DIR = BASE_DIR / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

## 1. ĐỌC DỮ LIỆU GỐC

In [3]:
df = pd.read_csv(RAW_PATH)
print("Shape ban đầu:", df.shape)

Shape ban đầu: (21321, 29)


## 2. LOẠI CỘT THIẾU QUÁ NHIỀU (>95%) VÀ CỘT KHÔNG CÓ BIẾN THIÊN

In [4]:
cols_to_drop = [
    'Instructions', 'Review', 'Cancellation / Rejection reason',
    'Restaurant compensation (Cancellation)', 'Restaurant penalty (Rejection)',
    'Customer complaint tag', 'City', 'Delivery'
]
actual_dropped = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=actual_dropped)
print(f"Đã loại {len(actual_dropped)} cột thiếu quá nhiều / không biến thiên")

Đã loại 8 cột thiếu quá nhiều / không biến thiên


## 3. PARSE THỜI GIAN

In [5]:
df['Order Placed At'] = pd.to_datetime(df['Order Placed At'], format='%I:%M %p, %B %d %Y', errors='coerce')
print(f"Số dòng không parse được thời gian: {df['Order Placed At'].isna().sum()}")
df['order_hour'] = df['Order Placed At'].dt.hour
df['order_dayofweek'] = df['Order Placed At'].dt.day_name()

# ------------------------------------------------------------
# 4. CHUẨN HÓA CỘT DISTANCE (text -> số km)
#    Quy ước: "<1km" -> 1km
# ------------------------------------------------------------
def parse_distance(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower().replace('km', '')
    if val.startswith('<'):
        return 1.0
    try:
        return float(val)
    except ValueError:
        return np.nan

df['distance_km'] = df['Distance'].apply(parse_distance)
print(f"Số dòng không parse được Distance: {df['distance_km'].isna().sum()}")

Số dòng không parse được thời gian: 0
Số dòng không parse được Distance: 0


## 5. XỬ LÝ CỘT DISCOUNT CONSTRUCT (missing = không áp dụng khuyến mãi)

In [6]:
df['Discount construct'] = df['Discount construct'].fillna('No Discount')

## 6. TẠO CỘT TỔNG KHUYẾN MÃI

In [7]:
discount_cols = [
    'Restaurant discount (Promo)',
    'Restaurant discount (Flat offs, Freebies & others)',
    'Gold discount',
    'Brand pack discount'
]
df['total_discount'] = df[discount_cols].sum(axis=1)
df['has_discount'] = df['total_discount'] > 0

## 7. XỬ LÝ RATING (thiếu 88.3%) - KHÔNG điền giá trị giả

In [8]:
print(f"\nSố đơn có Rating: {df['Rating'].notna().sum()} / {len(df)} "
      f"({df['Rating'].notna().mean()*100:.1f}%)")


Số đơn có Rating: 2491 / 21321 (11.7%)


## 8. KIỂM TRA OUTLIER

In [9]:
print("\nThống kê mô tả các cột số chính:")
print(df[['Bill subtotal', 'Total', 'distance_km', 'KPT duration (minutes)',
           'Rider wait time (minutes)']].describe())


Thống kê mô tả các cột số chính:
       Bill subtotal         Total   distance_km  KPT duration (minutes)  \
count   21321.000000  21321.000000  21321.000000            21026.000000   
mean      750.076838    682.616113      4.182731               17.332960   
std       498.759428    465.313977      2.977212                6.283388   
min        50.000000     52.500000      1.000000                0.000000   
25%       459.000000    387.450000      2.000000               13.380000   
50%       629.000000    597.450000      3.000000               16.330000   
75%       899.000000    837.900000      6.000000               20.050000   
max     16080.000000  12663.000000     21.000000               90.870000   

       Rider wait time (minutes)  
count               21153.000000  
mean                    4.825070  
std                     4.982591  
min                     0.100000  
25%                     1.000000  
50%                     3.100000  
75%                     7.400000  
m

## 9. LƯU FILE ĐÃ LÀM SẠCH

In [10]:
df.to_csv(CLEAN_DIR / 'zomato_clean.csv', index=False)
print(f"\nĐã lưu zomato_clean.csv | Shape cuối: {df.shape}")
print("=== HOÀN TẤT CLEANING ZOMATO ===")


Đã lưu zomato_clean.csv | Shape cuối: (21321, 26)
=== HOÀN TẤT CLEANING ZOMATO ===
